# FinanceBench-RAG

Grounded question-answering over SEC 10-K filings, benchmarked on the FinanceBench dataset.

Notebook structure follows [SPEC.md §11](../SPEC.md):

- **Setup** - imports, env, Nebius client, dataset load/filter, shared helpers.
- **Task 1** - Naive generation (no retrieval).
- **Task 2** - RAG theory write-up.
- **Task 3** - Embed documents.
- **Task 4** - `answer_with_rag` pipeline.
- **Task 5** - Run & compare.
- **Task 6** - Evaluation (correctness + faithfulness + page-hit@k).
- **Task 7** - Improvement cycles.
- **Bonus** - Multi-scale chunking.

In [16]:
# Install dependencies. Safe to re-run - pip will skip what's already satisfied.
# Mirrors requirements.txt; kept here so the notebook runs end-to-end without external setup.
%pip install -q \
    python-dotenv \
    openai \
    pandas \
    openpyxl \
    datasets \
    huggingface_hub \
    tqdm \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    pypdf \
    cryptography \
    sentence-transformers \
    ragas \
    requests

Note: you may need to restart the kernel to use updated packages.


## Setup

Imports, env loading, Nebius client, dataset load + filter, and shared helpers used across tasks.
Only the pieces needed by tasks already implemented are wired up - later setup cells will be added in their own task sections as we grow the notebook.

In [22]:
from __future__ import annotations

import json
import os
import re
import time
from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

load_dotenv(REPO_ROOT / ".env")

NEBIUS_API_KEY = os.environ["NEBIUS_API_KEY"]
NEBIUS_BASE_URL = "https://api.studio.nebius.ai/v1"

GENERATION_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
JUDGE_MODEL = "deepseek-ai/DeepSeek-V3.2"

client = OpenAI(api_key=NEBIUS_API_KEY, base_url=NEBIUS_BASE_URL)
print(f"Nebius client ready. Generation model: {GENERATION_MODEL}")

Nebius client ready. Generation model: meta-llama/Llama-3.3-70B-Instruct


In [3]:
def chat(
    messages: list[dict],
    model: str = GENERATION_MODEL,
    temperature: float = 0.0,
    max_tokens: int = 512,
    max_retries: int = 3,
) -> str:
    """Thin wrapper over Nebius chat completions with simple retry."""
    last_err: Exception | None = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return resp.choices[0].message.content or ""
        except Exception as exc:
            last_err = exc
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Nebius chat failed after {max_retries} retries: {last_err}")

smoke = chat(
    [{"role": "user", "content": "Reply with the single word: ready"}],
    max_tokens=8,
)
print(f"Smoke test → {smoke!r}")

Smoke test → 'ready'


### Dataset load, filter, and URL fix

Per [SPEC.md §3](../SPEC.md#3-dataset-contract):

1. Drop `question_type == "metrics-generated"` rows.
2. Replace dead `doc_link` URLs with the mirror repo: [patronus-ai/financebench](https://github.com/patronus-ai/financebench/tree/main/pdfs) (raw base: `https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs`).
3. `evidence_page_num` stays 0-indexed (we'll re-use this in Task 3 onward).

We hold the filtered DataFrame in `df` for the rest of the notebook.

In [4]:
from datasets import load_dataset

FINANCEBENCH_MIRROR = "https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs"

raw = load_dataset("PatronusAI/financebench", split="train")
df_full = raw.to_pandas()

df = df_full[df_full["question_type"] != "metrics-generated"].copy()
df["doc_link"] = df["doc_name"].apply(lambda n: f"{FINANCEBENCH_MIRROR}/{n}.pdf")
df = df.sort_values("financebench_id").reset_index(drop=True)

print(f"Full rows: {len(df_full)}")
print(f"After filter: {len(df)}")
print("Question types:", df["question_type"].value_counts().to_dict())
df.head(3)

Full rows: 150
After filter: 100
Question types: {'domain-relevant': 50, 'novel-generated': 50}


,financebench_id,company,doc_name,question_type,question_reasoning,domain_question_num,question,answer,justification,dataset_subset_label,evidence,gics_sector,doc_type,doc_period,doc_link
0,financebench_id_00005,Corning,CORNING_2022_10K,domain-relevant,Numerical reasoning OR Logical reasoning,dg24,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,"Trade accounts receivable, net of doubtful acc...",OPEN_SOURCE,[{'evidence_text': 'Consolidated Balance Sheet...,Information Technology,10k,2022,https://raw.githubusercontent.com/patronus-ai/...
1,financebench_id_00070,American Water Works,AMERICANWATERWORKS_2022_10K,domain-relevant,Numerical reasoning OR Logical reasoning,dg24,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",Accounts receivable+Income tax receivable+Unbi...,OPEN_SOURCE,[{'evidence_text': 'American Water Works Compa...,Utilities,10k,2022,https://raw.githubusercontent.com/patronus-ai/...
2,financebench_id_00080,Paypal,PAYPAL_2022_10K,domain-relevant,Numerical reasoning OR Logical reasoning,dg24,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,"Accounts receivable, net+Loans and interest re...",OPEN_SOURCE,"[{'evidence_text': 'PayPal Holdings, Inc. CONS...",Financials,10k,2022,https://raw.githubusercontent.com/patronus-ai/...


## Task 1 - Naive Generation

**Goal.** Hit Llama-3.3-70B-Instruct with each question directly - no retrieval, no context - and record what the model says.

**Sample.** First 5 questions of each `question_type` (sorted by `financebench_id`): 5 *domain-relevant* + 5 *novel-generated* = 10 questions.

**Verdict.** The assignment requires **manual judgment** - verdicts are filled in by hand after eyeballing each `(question, naive_answer, ground_truth)` row against the 4-way label space `{correct, partially correct, wrong, refused}`. The notebook writes the xlsx with the `verdict` column blank, ready to be filled in.

In [5]:
TASK1_N_PER_TYPE = 5

task1_df = (
    df.sort_values("financebench_id")
      .groupby("question_type", group_keys=False)
      .head(TASK1_N_PER_TYPE)
      .reset_index(drop=True)
)

print(f"Task 1 sample: {len(task1_df)} questions")
print(task1_df["question_type"].value_counts().to_dict())
task1_df[["financebench_id", "question_type", "question"]]

Task 1 sample: 10 questions
{'domain-relevant': 5, 'novel-generated': 5}


,financebench_id,question_type,question
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...


In [6]:
NAIVE_SYSTEM_PROMPT = (
    "You are a financial analyst answering questions about public companies. "
    "Answer concisely and directly. If you do not have enough information to answer, "
    "say so explicitly rather than guessing."
)

def naive_answer(question: str) -> str:
    return chat(
        [
            {"role": "system", "content": NAIVE_SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        model=GENERATION_MODEL,
        temperature=0.0,
        max_tokens=400,
    )

naive_answers: list[str] = []
for q in tqdm(task1_df["question"].tolist(), desc="Naive generation"):
    naive_answers.append(naive_answer(q))

task1_df["naive_answer"] = naive_answers
task1_df[["financebench_id", "question_type", "question", "naive_answer"]].head()

Naive generation: 100%|██████████| 10/10 [00:48<00:00,  4.87s/it]


,financebench_id,question_type,question,naive_answer
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,"Based on Corning's FY2022 data, the company's ..."
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"Based on American Water Works' FY2022 data, I ..."
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,"Based on PayPal's FY2022 data, the company's c..."
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,Gross margins are not a relevant metric for JP...
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,"Yes, based on FY 2022 data, Verizon is conside..."


In [7]:
from datetime import datetime

task1_out = task1_df.rename(columns={"answer": "ground_truth"}).copy()
task1_out["verdict"] = ""  # filled in manually after review (see Discussion below)

task1_out = task1_out[
    [
        "financebench_id",
        "question_type",
        "question",
        "naive_answer",
        "ground_truth",
        "verdict",
    ]
]

_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
task1_path = ARTIFACTS_DIR / f"assignment2_naive_generation_{_ts}.xlsx"
task1_out.to_excel(task1_path, index=False)
print(f"Wrote {task1_path.relative_to(REPO_ROOT)}  ({len(task1_out)} rows)")
print("Fill in the 'verdict' column, then save a copy named assignment2_naive_generation.xlsx for submission.")
task1_out

Wrote artifacts/assignment2_naive_generation_20260427_211338.xlsx  (10 rows)
Fill in the 'verdict' column, then save a copy named assignment2_naive_generation.xlsx for submission.


,financebench_id,question_type,question,naive_answer,ground_truth,verdict
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,"Based on Corning's FY2022 data, the company's ...",Yes. Corning had a positive working capital am...,
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"Based on American Water Works' FY2022 data, I ...","No, American Water Works had negative working ...",
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,"Based on PayPal's FY2022 data, the company's c...",Yes. Paypal has a positive working capital of ...,
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,Gross margins are not a relevant metric for JP...,"Since JPM is a financial institution, gross ma...",
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,"Yes, based on FY 2022 data, Verizon is conside...",Yes. Verizon's capital intensity ratio was app...,
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,I don't have the most current information on P...,77.78,
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,I don't have enough information to answer that...,"Yes, there was a decline of ~42% between FY202...",
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,I don't have access to JPM's 2021 Q1 financial...,Corporate. Its net revenue was -$473 million.,
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,I don't have the specific information on Pfize...,"Yes, change in PPNE was positive year over year",
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,I don't have the specific information on MGM's...,Las Vegas resorts contributed ~90% of company ...,


### Task 1 - Discussion

**Verdict distribution (manually judged, 4-way label space `{correct, partially correct, wrong, refused}`):**

| Verdict           | domain-relevant | novel-generated | Total |
|-------------------|-----------------|-----------------|-------|
| correct           | 2               | 0               | 2     |
| partially correct | 1               | 0               | 1     |
| wrong             | 1               | 0               | 1     |
| refused           | 1               | 5               | 6     |
| **Total**         | **5**           | **5**           | **10**|

**1. When did the model refuse to answer? Why do you think it refused?**

The model refused on 6 of 10 questions: 1 domain-relevant (American Water Works working capital) and all 5 novel-generated questions.

The refusals share the same shape: the model says it lacks specific company financials for the requested fiscal period and redirects the user to the company's filings or investor pages.

The novel-generated questions ask for exact figures or specific segment / region rankings (e.g., "How much does Pfizer expect to pay to spin off Upjohn in USD million?", "Which of JPM's business segments had the lowest net revenue in 2021 Q1?"), which the model cannot produce from parametric memory alone, so refusal is the safe choice.

The American Water Works refusal is the same pattern: a numeric balance sheet question on a less prominent issuer that the model has not memorized precisely.

Refusals correlate strongly with question specificity (exact numbers, narrow time windows, lesser-known issuers) rather than with `question_type` per se.

**2. When did the model answer confidently?**

The model answered confidently on 4 of 5 domain-relevant questions, all of which had a binary or qualitative shape (yes/no on working capital, yes/no on capital intensity, "is gross margin relevant for a bank") rather than asking for a precise figure.

Of those 4 confident answers:

- **2 correct:** JPM gross margin not relevant (correctly identified as a financial institution); Verizon capital intensive (correct yes/no with reasonable supporting reasoning on network infrastructure).
- **1 partially correct:** Corning working capital - right direction (positive) but wrong magnitude, because the model used full current assets / liabilities ($3.7B) instead of operating-only items ($831M ground truth).
- **1 wrong:** PayPal working capital - opposite sign vs. ground truth (model said negative -$4.5B, ground truth is positive $1.6B).

Confidence tracked with question shape: when the answer space is small and the issuer is well known, the model produces a fluent answer even when the supporting numbers are hallucinated.

This is the dangerous failure mode: a refusal is visible and easy to triage, but a confident answer paired with hallucinated supporting numbers looks authoritative and slips past unless someone checks the filing. RAG is built for exactly this case - grounding the supporting numbers in retrieved 10-K text turns a confident-sounding answer into a verifiable one.

**3. Are there patterns related to `question_type`?**

Yes, the split is sharp.

Domain-relevant questions (curated by FinanceBench) skew toward qualitative or binary judgments on prominent large-cap issuers; the model engaged on 4/5 with mixed accuracy.

Novel-generated questions skew toward precise figures, narrow time windows (single quarters, year-over-year deltas), and specific segment or region rankings; the model refused on 5/5.

Two failure modes are worth flagging for the later RAG comparison: (a) silent hallucination on confident answers (PayPal numbers, Corning magnitude) where the model produces plausible-looking figures that are wrong, and (b) blanket refusal on any question requiring an exact number it has not memorized.

RAG should help with both: grounding answers in retrieved 10-K text should reduce hallucination on the confident answers, and supplying the relevant filing should let the model attempt the questions it currently refuses.


## Task 2 - RAG Reminder

A simple RAG pipeline has three components: **indexing** (documents into a searchable vector store), **retrieval** (query into top-k relevant chunks), and **generation** (query + chunks into a grounded answer). For each component, three questions: what does it contribute, where can it fail, and does it run once offline or per query.

### Indexing

**1. Contribution.** Indexing turns a static document corpus into something queryable by meaning rather than keywords. Typically PDFs are loaded, split into chunks (here, 1000 chars with 150 overlap), embedded with a sentence-transformer model (we use `BAAI/bge-small-en-v1.5`), and stored in a vector index (FAISS in our case) alongside source metadata such as `doc_name` and `page_number`.

**2. Failure modes.** Bad chunk boundaries are the most common failure - e.g., splitting a balance-sheet row so the number survives but its label "Total current liabilities" is lost. Embedding model mismatch with the domain (a general-web sentence-transformer may not separate fiscal-year variants of the same financial metric well) and metadata mistakes (1-indexed vs. 0-indexed page numbers) also break things downstream.

**3. When it runs.** **Offline, once.** The index is built up front, persisted to disk, and reused across many queries. It is only rebuilt when the corpus or the chunking / embedding strategy changes.

### Retrieval

**1. Contribution.** Retrieval embeds the user question with the *same* model used at index time, then pulls the top-k nearest chunks from the vector index (FAISS in our case) using the index's similarity metric. This narrows the corpus from thousands of chunks down to a few that fit in the generator's context window, and it decides whether the right evidence is even available to the generator.

**2. Failure modes.** A recall miss - the evidence page is not in the top-k because k is too small or the question's wording is semantically far from the chunk's. The opposite also bites: top-k surfaces plausible-but-wrong chunks that the generator anchors to, e.g., FY2021 chunks for an FY2022 question, or a Merck filing chunk for a Pfizer question because both mention Upjohn-style spinoffs.

**3. When it runs.** **Online, per query.** Each user question triggers a fresh embed-and-search round trip against the index.

### Generation

**1. Contribution.** Generation takes the user query plus the retrieved chunks (formatted with clear separators and source metadata such as `doc_name`) and asks an LLM (we use `Llama-3.3-70B-Instruct`) to produce an answer grounded in those chunks rather than its parametric memory. The prompt also instructs the model to refuse when the chunks do not support an answer, which is what protects against the hallucination failure mode we saw in Task 1.

**2. Failure modes.** Ignoring the context and falling back to memorized knowledge (a faithfulness failure - the answer "looks right" but is not actually supported by the chunks). Misreading the chunks (mixing up two fiscal-year columns in the same chunk, picking the wrong row from a table). Miscalibrated refusals - refusing when the answer is actually present (over-cautious) or answering when the chunks are irrelevant (under-cautious).

**3. When it runs.** **Online, per query.** Each question gets its own generation call with its own freshly retrieved context.


## Task 3 - Embed Documents

Build the searchable index that the RAG pipeline retrieves from. Three steps: download the PDFs referenced by the filtered dataset, load and chunk them with page-level metadata, then embed and persist to a FAISS vector store. The index is built once and cached on disk so later tasks (and re-runs) load it instead of rebuilding.

Baseline configuration (per [SPEC.md §4.1](../SPEC.md#41-indexing-offline-once-per-chunk-configuration)): `chunk_size=1000`, `chunk_overlap=150`, `BAAI/bge-small-en-v1.5` embeddings. Metadata kept on every chunk: `doc_name`, `company`, `doc_period`, `page_number` (0-indexed, matching the dataset's `evidence_page_num` convention).

In [8]:
# Download the PDFs referenced by the filtered dataset. Idempotent: skips files already on disk.
import requests

PDF_DIR = REPO_ROOT / "data" / "pdfs"
PDF_DIR.mkdir(parents=True, exist_ok=True)

doc_names = sorted(df["doc_name"].unique())
to_download = [n for n in doc_names if not (PDF_DIR / f"{n}.pdf").exists()]

print(f"Unique docs in filtered dataset: {len(doc_names)}")
print(f"Already cached: {len(doc_names) - len(to_download)}; to download: {len(to_download)}")

for name in tqdm(to_download, desc="Downloading PDFs"):
    url = f"{FINANCEBENCH_MIRROR}/{name}.pdf"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    (PDF_DIR / f"{name}.pdf").write_bytes(resp.content)

local = sorted(p.name for p in PDF_DIR.glob("*.pdf"))
print(f"Local PDFs after download: {len(local)} (expected {len(doc_names)})")
assert len(local) == len(doc_names), "PDF count mismatch - investigate before building the index."


Unique docs in filtered dataset: 42
Already cached: 42; to download: 0


Local PDFs after download: 42 (expected 42)


In [9]:
# Build (or load) the FAISS index. Loader -> page-level metadata -> recursive chunking -> bge embeddings -> FAISS.
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
INDEX_DIR = REPO_ROOT / "indices" / f"faiss_chunk{CHUNK_SIZE}"

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

if INDEX_DIR.exists():
    print(f"Loading cached index from {INDEX_DIR.relative_to(REPO_ROOT)}")
    vectorstore = FAISS.load_local(
        str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True
    )
else:
    print(f"Building FAISS index at {INDEX_DIR.relative_to(REPO_ROOT)}")
    doc_meta = (
        df.drop_duplicates("doc_name")
          .set_index("doc_name")[["company", "doc_period"]]
          .to_dict("index")
    )

    page_docs = []
    for name in tqdm(doc_names, desc="Loading PDFs"):
        loader = PyPDFLoader(str(PDF_DIR / f"{name}.pdf"))
        for i, page in enumerate(loader.load()):
            page.metadata = {
                "doc_name": name,
                "company": doc_meta[name]["company"],
                "doc_period": doc_meta[name]["doc_period"],
                "page_number": i,  # 0-indexed, matches dataset's evidence_page_num
            }
            page_docs.append(page)
    print(f"Loaded {len(page_docs)} pages from {len(doc_names)} PDFs.")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
    )
    chunks = splitter.split_documents(page_docs)
    print(f"Split into {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}).")

    vectorstore = FAISS.from_documents(chunks, embeddings)
    INDEX_DIR.parent.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(INDEX_DIR))
    print(f"Saved index to {INDEX_DIR.relative_to(REPO_ROOT)}")

print(f"Index size: {vectorstore.index.ntotal} vectors")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9104.62it/s]


Loading cached index from indices/faiss_chunk1000
Index size: 24218 vectors


In [10]:
# Spot-check retrieval on a few questions: do the top-k chunks land on or near the labeled evidence pages?
import ast

def _evidence_pages(row) -> list[int]:
    """Return the set of evidence page numbers that belong to row['doc_name'].

    The dataset's schema permits cross-doc evidence (each evidence entry carries
    its own doc_name); empirically this filtered set never uses it, but we filter
    defensively to avoid page-number collisions across docs if that ever changes.
    """
    ev = row["evidence"]
    if isinstance(ev, str):
        ev = ast.literal_eval(ev)
    return sorted({
        int(e["evidence_page_num"])
        for e in ev
        if e.get("doc_name", row["doc_name"]) == row["doc_name"]
    })

SPOT_CHECK_IDS = [
    "financebench_id_00005",  # Corning working capital (domain-relevant)
    "financebench_id_00206",  # JPM gross margin (domain-relevant)
    "financebench_id_00283",  # Pfizer Upjohn spinoff (novel-generated)
]

for fid in SPOT_CHECK_IDS:
    row = df.loc[df["financebench_id"] == fid].iloc[0]
    ev_pages = _evidence_pages(row)
    print("=" * 88)
    print(f"{fid}  ({row['question_type']})  doc={row['doc_name']}  evidence_pages={ev_pages}")
    print(f"Q: {row['question']}")
    print("-" * 88)
    hits = vectorstore.similarity_search(row["question"], k=4)
    for i, d in enumerate(hits, 1):
        same_doc = d.metadata["doc_name"] == row["doc_name"]
        page_hit = d.metadata["page_number"] in ev_pages
        if same_doc and page_hit:
            flag = "HIT_doc_and_pg"
        elif same_doc:
            flag = "same_doc      "
        else:
            flag = "              "
        snippet = d.page_content.replace("\n", " ")[:160]
        print(f"  [{flag}] k={i}  {d.metadata['doc_name']}  p{d.metadata['page_number']}  | {snippet}")
    print()


financebench_id_00005  (domain-relevant)  doc=CORNING_2022_10K  evidence_pages=[59]
Q: Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.
----------------------------------------------------------------------------------------
  [same_doc      ] k=1  CORNING_2022_10K  p101  | (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since
  [same_doc      ] k=2  CORNING_2022_10K  p102  | (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since
  [              ] k=3  3M_2022_10K  p37  | not defined under U.S. generally accepted accounting principles and may not be computed the same as similarly titled measures used by other companies. The Compa
  [              ] k

### Task 3 - Discussion

**Spot-check observations.** Three sample questions probed against the index, summarized below; per-row detail and the per-axis read follow. We use top-k = 4 here to match the Task 4 default `answer_with_rag(k=4)`; the brief leaves Task 3's k open.

| Question                       | Evidence doc          | Evidence page(s) | Right document? @ k=4   | Right page? @ k=4 | Top chunks close to evidence text? |
|--------------------------------|-----------------------|------------------|--------------------------|--------------------|-------------------------------------|
| Corning working capital        | `CORNING_2022_10K`    | 59               | 2/4                      | 0/4                | No, but k=4 is a 3M working-capital table (right shape, wrong issuer) |
| JPM gross margin               | `JPMORGAN_2022_10K`   | 2                | 0/4 (1/4 right issuer, wrong filing) | 0/4    | No (MD&A boilerplate)               |
| Pfizer Upjohn future payment   | `Pfizer_2023Q2_10Q`   | 40               | 0/4 (4/4 right issuer, wrong filing) | 0/4    | No (2020-era Upjohn mechanics)      |

Read on each axis:

- **Right document?** Mixed to poor. Corning landed 2/4 chunks inside `CORNING_2022_10K`; JPM hit 0/4 inside `JPMORGAN_2022_10K` (k=2 is `JPMORGAN_2022Q2_10Q`, the right issuer but the wrong filing); Pfizer hit 0/4 inside the evidence doc `Pfizer_2023Q2_10Q`, although all 4 of its chunks came from `PFIZER_2021_10K` (same issuer, wrong filing).
- **Close to the evidence text?** Not really. None of the top-4 chunks cover the content the evidence pages actually contain (Corning's consolidated balance sheet, JPM's gross-margin discussion, Pfizer's future spin-off payment commitment). The closest near-miss is the Corning case: k=4 surfaces 3M's working-capital table (current assets minus current liabilities), which is structurally what we wanted but from the wrong issuer. JPM's chunks are MD&A boilerplate (a BestBuy disclaimer at k=1, a JPM 10-Q metrics table at k=2, AMD and Boeing accounting-estimate language at k=3 and k=4); Pfizer's chunks are 2020-era Upjohn spin-off mechanics rather than the forward-looking number the question asks for.
- **Right page number?** Zero `HIT_doc_and_pg` across the three questions. `page_hit@4` on this small sample is 0/3.

Per-row detail:

- **Corning** (evidence p59, consolidated balance sheet). k=1 and k=2 stayed inside `CORNING_2022_10K` but landed on pages 101 and 102 (the HSG consolidation note, near-identical lead text on both pages), not p59. k=3 is `3M_2022_10K` p37, non-GAAP measures definition language. k=4 is `3M_2023Q2_10Q` p70, an actual working-capital table (current assets, current liabilities, working-capital line) for 3M. The embedder pulled the right *shape* of evidence into the top-4, but from the wrong company; meanwhile the same-doc chunks it found were Corning footnote material rather than the balance sheet.
- **JPM gross margin** (evidence p2): top-1 is a BestBuy 10-K disclaimer paragraph; k=2 is `JPMORGAN_2022Q2_10Q` (right issuer, wrong filing: a 10-Q rather than the labeled 10-K); k=3 and k=4 are AMD and Boeing MD&A boilerplate. The question's "gross margins... fluctuating" phrasing maps to generic risk-factor language that exists in many 10-Ks, and "JPM" is too short to dominate the embedding.
- **Pfizer Upjohn** (evidence p40 in `Pfizer_2023Q2_10Q`, future spin-off cost). All 4 chunks come from `PFIZER_2021_10K`. k=1, k=2, k=4 discuss the Upjohn spin-off as past history (a 2020 cash-flow note, the "Transforming to a More Focused Company" narrative, and the VTRS NASDAQ listing detail). k=3 is a Pfizer revenue / alliance / biosimilars table where the Consumer Healthcare line shows post-spin-off zeros. None contain the forward-looking payment commitment the question asks for. The embedder picked the older filing that talks about Upjohn the most, missing the newer 10-Q.

**Useful weak baseline.** Three distinct failure shapes appeared, each mapping cleanly to a Task 7 experiment:

- **Right doc, wrong page** (Corning). Retrieval found the correct issuer's 10-K but landed on a footnote rather than the balance sheet. *Lever:* increase `k` (E1) so the relevant page has more chances to appear in the top-k.
- **Issuer confusion under generic question wording** (JPM). The question's vocabulary mapped to MD&A boilerplate that exists across many 10-Ks; the issuer signal was too weak to dominate. *Lever:* a cross-encoder reranker (E3) or a stricter prompt (E4) to penalize chunks that are not from the named issuer.
- **Filing-year confusion** (Pfizer). The embedder picked the older filing where the entity is discussed most often, missing the newer filing that holds the forward-looking number. *Levers:* smaller chunks (E2) so the embedder commits to less content per chunk, plus metadata filtering by `doc_period` if needed.

---

**Why these settings.** Baseline `chunk_size=1000` with `chunk_overlap=150` is a balance: small enough that a single chunk is roughly one topic in a 10-K (a paragraph or a small table block) and the embedding stays sharp, large enough that a balance-sheet line and its label usually survive in the same chunk. The 150-char overlap reduces the chance that a fact gets cut at the seam. Bonus task explores 300 and 2000 to see whether this baseline is actually best for FinanceBench.

**Why `BAAI/bge-small-en-v1.5`.** Small (384-dim, ~130 MB), strong on English retrieval benchmarks, and CPU-friendly so the index builds in single-digit minutes on a laptop. Good fit for a baseline; if we see the embedder confuse fiscal-year variants of the same metric we can swap to a larger BGE or a finance-specific encoder later.

**Why FAISS.** Local, file-backed, no service to run. `save_local` produces a directory we can re-load in any later cell or experiment, which is exactly the persistence story the spec calls for.

**Page metadata.** We attach our own `page_number` field set to the loader's enumeration index, deliberately overwriting `PyPDFLoader`'s default `page` field. The dataset's `evidence_page_num` is 0-indexed; we keep our metadata 0-indexed too so `page_hit@k` (Task 6) is a direct equality check with no off-by-one.


## Task 4 - `answer_with_rag` pipeline

Wraps Task 3's index into a single function `answer_with_rag(query, k=4) -> dict`. The pipeline is: embed the query with the same `BAAI/bge-small-en-v1.5` model used at index time, retrieve the top-k chunks from FAISS, format them as a labeled context block, and call Llama-3.3-70B-Instruct with a system prompt that constrains the answer to the retrieved context.

**Prompt design (per [SPEC.md §4.3](../SPEC.md#43-generation)):**

- **Chunk formatting.** Each retrieved chunk is prefixed with `doc_name` and `page_number` and separated from the next by `\n\n---\n\n`. The doc_name lets the model cite per fact; the explicit separator stops the model from blending two chunks into one paragraph.
- **Why `\n\n---\n\n` as the separator.** The blank lines are required: `---` only parses as a markdown horizontal rule (a hard break the LLM recognizes) when surrounded by them. Cheap (~4 tokens), unambiguous (never appears inside 10-K prose).
- **Empty retrieval.** If FAISS returns zero chunks, the context block is replaced with the sentinel `"No relevant context was found."` rather than an empty string. This forces the model to take the "context does not contain the answer" branch instead of hallucinating from parametric memory.
- **System prompt** instructs the model to (a) answer only from context, (b) say so explicitly when context is insufficient, (c) keep answers concise and cite the `doc_name` per fact.
- **Temperature 0** for reproducibility across reruns.

The returned dict carries the `answer` plus the `retrieved_chunks` (each with its `text`, `doc_name`, `page_number`), which Task 5 reuses without re-running retrieval.

In [11]:
# answer_with_rag pipeline: query -> retrieve -> format context -> generate -> grounded answer.
RAG_SYSTEM_PROMPT = (
    "You are a financial analyst answering questions about public company filings.\n"
    "Answer using ONLY the provided context. Do not rely on prior knowledge.\n"
    "If the context does not contain the information needed to answer, say so explicitly "
    "rather than guessing.\n"
    "Keep answers concise. For each fact in your answer, cite the document it came from "
    "in parentheses, for example: (CORNING_2022_10K)."
)

EMPTY_CONTEXT_SENTINEL = "No relevant context was found."
CHUNK_SEPARATOR = "\n\n---\n\n"

def format_context_block(chunks: list) -> str:
    """Format retrieved chunks as labeled blocks separated by ---. Empty input returns a sentinel."""
    if not chunks:
        return EMPTY_CONTEXT_SENTINEL
    parts = []
    for i, c in enumerate(chunks, 1):
        header = (
            f"[chunk {i}] doc_name={c.metadata['doc_name']} "
            f"page={c.metadata['page_number']}"
        )
        parts.append(f"{header}\n{c.page_content}")
    return CHUNK_SEPARATOR.join(parts)

def answer_with_rag(query: str, k: int = 4) -> dict:
    """Retrieve top-k chunks for `query`, ground a Llama-3.3 answer in them, and return both."""
    chunks = vectorstore.similarity_search(query, k=k)
    context_block = format_context_block(chunks)
    user_message = f"Question: {query}\n\nContext:\n{context_block}"
    answer = chat(
        [
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        model=GENERATION_MODEL,
        temperature=0.0,
        max_tokens=500,
    )
    retrieved_chunks = [
        {
            "text": c.page_content,
            "doc_name": c.metadata["doc_name"],
            "page_number": c.metadata["page_number"],
        }
        for c in chunks
    ]
    return {"answer": answer, "retrieved_chunks": retrieved_chunks}

print("answer_with_rag defined. Signature: answer_with_rag(query: str, k: int = 4) -> dict")


answer_with_rag defined. Signature: answer_with_rag(query: str, k: int = 4) -> dict


### Example - chunk formatting in the context block

Concrete illustration of the prompt-design rules above. Runs `format_context_block` on a small retrieval (k=2, each chunk truncated to ~250 chars for readability) and prints the resulting block. This is exactly what gets wrapped into the user message and sent to the LLM. The empty-retrieval sentinel is shown on the last line for completeness.

In [14]:
# Build a sample context block (k=2 for readability, each chunk truncated to ~250 chars).
# This is exactly what the user message wraps before being sent to the LLM.
preview_question = "Does Corning have positive working capital based on FY2022 data?"
preview_chunks = vectorstore.similarity_search(preview_question, k=2)

class _Preview:
    def __init__(self, doc):
        self.metadata = doc.metadata
        self.page_content = (
            doc.page_content[:250] + " ... [truncated for preview]"
            if len(doc.page_content) > 250
            else doc.page_content
        )

preview_block = format_context_block([_Preview(c) for c in preview_chunks])

print(f"Question: {preview_question}\n")
print("=" * 88)
print("Context block (what the LLM sees inside the user message):")
print("=" * 88)
print(preview_block)
print("=" * 88)
print(f"\nEmpty-retrieval branch returns the sentinel: {format_context_block([])!r}")


Question: Does Corning have positive working capital based on FY2022 data?

Context block (what the LLM sees inside the user message):
[chunk 1] doc_name=CORNING_2022_10K page=101
(1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since September 9, 2020.  Refer to Note 3 (HSGTransactions and Acquisitions) in the notes to th ... [truncated for preview]

---

[chunk 2] doc_name=CORNING_2022_10K page=82
Based on borrowing rates currently available to us for loans with similar terms and maturities, the fair value of long-term debt was $6.1 billion and $8.3 billion as of December 31, 2022 and 2021, respectively, compared to recorded book values of $6. ... [truncated for preview]

Empty-retrieval branch returns the sentinel: 'No relevant context was found.'


In [13]:
# Demo: run answer_with_rag on the Task 3 spot-check questions and one synthetic empty-retrieval probe.
DEMO_IDS = [
    "financebench_id_00005",  # Corning working capital
    "financebench_id_00206",  # JPM gross margin
    "financebench_id_00283",  # Pfizer Upjohn future payment
]

for fid in DEMO_IDS:
    row = df.loc[df["financebench_id"] == fid].iloc[0]
    print("=" * 88)
    print(f"{fid}  evidence_doc={row['doc_name']}")
    print(f"Q: {row['question']}")
    print("-" * 88)
    result = answer_with_rag(row["question"], k=4)
    print(f"A: {result['answer']}")
    print()
    print("Retrieved chunks:")
    for i, c in enumerate(result["retrieved_chunks"], 1):
        print(f"  [{i}] {c['doc_name']}  p{c['page_number']}")
    print()

# Empty-retrieval probe: format_context_block on an empty list goes through the sentinel branch.
print("=" * 88)
print("Empty-retrieval probe (formatting only, no LLM call):")
print(format_context_block([]))


financebench_id_00005  evidence_doc=CORNING_2022_10K
Q: Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.
----------------------------------------------------------------------------------------
A: The provided context does not contain the information needed to answer whether Corning has positive working capital based on FY2022 data. The context only provides information about Corning's segments, transactions, and acquisitions, but does not include the company's current assets or liabilities, which are necessary to calculate working capital (CORNING_2022_10K). The information about working capital is provided for 3M, a different company (3M_2022_10K, 3M_2023Q2_10Q).

Retrieved chunks:
  [1] CORNING_2022_10K  p101
  [2] CORNING_2022_10K  p102
  [3] 3M_2022_10K  p37
  [4] 3M_2023Q2_10Q  p70

financebench_id_00206  evidence_doc=JPMORGAN_2022_10K
Q: Are JPM's gross

## Task 5 - Run & Compare

**Goal.** Run the same 10 Task 1 questions through `answer_with_rag` and put the naive answer, the RAG answer, and the ground truth side by side. Output is `artifacts/assignment2_run_and_compare.xlsx` with columns `financebench_id | question_type | question | naive_answer | RAG_answer | ground_truth`.

**Judgment.** No verdict column in the xlsx itself. The discussion below addresses the three brief questions: where did RAG help (refusals or hallucinations becoming grounded answers), where did RAG hurt (naive happened to be right, RAG retrieved the wrong filing or got confused), and patterns by `question_type`. Manual judgment, same style as Task 1: rows are reviewed by hand, conclusions written into the discussion cell.

In [15]:
# Run the 10 Task 1 questions through answer_with_rag and build the side-by-side comparison.
rag_answers: list[str] = []
rag_chunks_per_q: list[list] = []
for q in tqdm(task1_df["question"].tolist(), desc="RAG generation"):
    result = answer_with_rag(q, k=4)
    rag_answers.append(result["answer"])
    rag_chunks_per_q.append(result["retrieved_chunks"])

task5_out = task1_df.rename(columns={"answer": "ground_truth"}).copy()
task5_out["RAG_answer"] = rag_answers
task5_out = task5_out[
    [
        "financebench_id",
        "question_type",
        "question",
        "naive_answer",
        "RAG_answer",
        "ground_truth",
    ]
]

_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
task5_path = ARTIFACTS_DIR / f"assignment2_run_and_compare_{_ts}.xlsx"
task5_out.to_excel(task5_path, index=False)
print(f"Wrote {task5_path.relative_to(REPO_ROOT)}  ({len(task5_out)} rows)")
print("Save a copy named assignment2_run_and_compare.xlsx for submission.")
task5_out


RAG generation: 100%|██████████| 10/10 [00:23<00:00,  2.40s/it]

Wrote artifacts/assignment2_run_and_compare_20260427_214734.xlsx  (10 rows)
Save a copy named assignment2_run_and_compare.xlsx for submission.


,financebench_id,question_type,question,naive_answer,RAG_answer,ground_truth
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,"Based on Corning's FY2022 data, the company's ...",The provided context does not contain the info...,Yes. Corning had a positive working capital am...
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"Based on American Water Works' FY2022 data, I ...",The provided context does not contain the info...,"No, American Water Works had negative working ..."
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,"Based on PayPal's FY2022 data, the company's c...",The provided context does not contain informat...,Yes. Paypal has a positive working capital of ...
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,Gross margins are not a relevant metric for JP...,Gross margins are not a relevant metric for a ...,"Since JPM is a financial institution, gross ma..."
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,"Yes, based on FY 2022 data, Verizon is conside...","Based on FY 2022 data, Verizon's capital expen...",Yes. Verizon's capital intensity ratio was app...
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,I don't have the most current information on P...,"Pfizer expects to pay $1.6 billion, primarily ...",77.78
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,I don't have enough information to answer that...,"Yes, there was a drop in Cash & Cash equivalen...","Yes, there was a decline of ~42% between FY202..."
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,I don't have access to JPM's 2021 Q1 financial...,The business segment with the lowest net reven...,Corporate. Its net revenue was -$473 million.
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,I don't have the specific information on Pfize...,The context does not contain the information n...,"Yes, change in PPNE was positive year over year"
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,I don't have the specific information on MGM's...,The region with the highest EBITDAR contributi...,Las Vegas resorts contributed ~90% of company ...


### Task 5 - Discussion

**Per-row verdict log (manual judgment, 4-way label space `{correct, partially correct, wrong, refused}`):**

| #  | financebench_id | question_type    | Question (short)                              | Naive verdict      | RAG verdict | RAG helped / hurt / neutral |
|----|-----------------|------------------|-----------------------------------------------|--------------------|-------------|------------------------------|
| 0  | 00005           | domain-relevant  | Corning working capital FY22                  | partially correct  | refused     | helped                       |
| 1  | 00070           | domain-relevant  | American Water Works working capital FY22     | refused            | refused     | hurt (mild)                  |
| 2  | 00080           | domain-relevant  | PayPal working capital FY22                   | wrong              | refused     | helped                       |
| 3  | 00206           | domain-relevant  | JPM gross margin consistency                  | correct            | correct     | neutral                      |
| 4  | 00215           | domain-relevant  | Verizon capital intensive FY22                | correct            | correct     | helped (mild)                |
| 5  | 00283           | novel-generated  | Pfizer Upjohn future spin-off cost            | refused            | wrong       | hurt                         |
| 6  | 00288           | novel-generated  | Cash & equivalents drop FY23 to Q2 FY24       | refused            | correct     | helped                       |
| 7  | 00299           | novel-generated  | JPM lowest-revenue segment 2021 Q1            | refused            | wrong       | hurt                         |
| 8  | 00302           | novel-generated  | Pfizer PPNE growth FY20 to FY21               | refused            | refused     | neutral                      |
| 9  | 00382           | novel-generated  | MGM highest-EBITDAR region FY22               | refused            | correct     | helped                       |

**Tally (helped / hurt / neutral):** 5 helped, 3 hurt, 2 neutral.

**Tally by `question_type`:**

| question_type    | helped | hurt | neutral | total |
|------------------|--------|------|---------|-------|
| domain-relevant  | 3      | 1    | 1       | 5     |
| novel-generated  | 2      | 2    | 1       | 5     |
| **Total**        | **5**  | **3**| **2**   | **10**|

---

**1. Did RAG help?**

Yes, on 5 of 10 rows. Two distinct patterns of help:

- **Anti-hallucination on confident-but-wrong naive answers** (rows 0, 2). Naive gave Corning's working capital as $3.7B (right sign, wrong magnitude: used full current assets / liabilities instead of operating-only items) and PayPal's as -$4.5B (opposite sign vs. ground truth). RAG refused both because the retrieved chunks did not contain the relevant balance-sheet line items. RAG didn't produce the right answer, but it stopped the user from being handed a misleading number with confidence. In the PayPal case especially, naive would have been actively harmful.
- **Refused-to-correct on questions that require corpus access** (rows 6, 9). The Best Buy cash drop and the MGM EBITDAR-by-region questions are unanswerable without the filing; naive correctly refused both. RAG retrieved the right filings, computed (or read off) the correct answer, and cited the source. Row 6 is the cleanest demo: the question doesn't even name the company, but retrieval surfaced both `BESTBUY_2023_10K` and `BESTBUY_2024Q2_10Q`, the generator pulled the cash positions from each, and the implied ~42% decline matches the ground truth.
- **Citation grounding on already-correct naive** (row 4). Verizon's "yes, capital-intensive" was already correct in Task 1, but RAG re-grounded the supporting capex figure ($23.1B) in `VERIZON_2022_10K`. Same conclusion, but now traceable.

**2. Did RAG hurt?**

Yes, on 3 of 10 rows. One dominant failure shape:

- **Shifted failure mode from clean refusal to confident-with-wrong-citation** (rows 5, 7). The Pfizer Upjohn question and the JPM lowest-revenue-segment question are both novel-generated. Naive cleanly refused both. RAG produced confident answers with citations that look legitimate but are wrong on both the number and the source. Pfizer: $1.6B from `PFIZER_2021_10K` p75 instead of $77.78M from `Pfizer_2023Q2_10Q` p40 (filing-year confusion; the embedder picked the older filing where Upjohn is discussed at length, missing the newer 10-Q with the forward-looking number). JPM: Commercial Banking $2,483M from a 2022 Q2 10-Q instead of Corporate -$473M for 2021 Q1 (wrong period entirely, plus missed the negative-revenue segment that makes the question's "lowest" non-obvious). This is the most dangerous pattern in the run: a user without access to the ground truth has no signal that the answer is wrong, and the citation makes it look authoritative.
- **Layered-on incorrect inference** (row 1, mild). Both naive and RAG refused on the actual American Water Works working-capital question, but RAG added an unsupported claim that "working capital may not be useful for a utility company"; self-flagged as inference, but the ground truth is that the company has a definite, sizable working-capital figure (-$1,561M), so the inference points the user *away* from looking it up. Naive's refusal was cleaner.

The two hard `hurt` cases (5 and 7) trace back to the failure shapes called out in the Task 3 spot-check discussion: filing-year confusion when an issuer has multiple filings in the index, and retrieval bringing back content from the wrong period.

**3. Patterns by `question_type`**

The two question types behave differently, and this is the sharpest signal in the run:

- **Domain-relevant (3 helped, 1 hurt mild, 1 neutral).** These questions are qualitative or yes/no judgments on prominent issuers (working capital direction, capital intensity, gross-margin relevance for a bank). Naive engaged on 4/5 (only American Water Works refused), with mixed accuracy. RAG's role here is mostly **damage control on confident-but-wrong naive answers**: when naive hallucinated a number with confidence (Corning, PayPal), RAG's "answer only from context" constraint converted that into an honest refusal. When naive happened to be right (JPM gross margin, Verizon capital intensity), RAG matched it or added a citation. Net positive, with the wins concentrated in the anti-hallucination role.
- **Novel-generated (2 helped, 2 hurt, 1 neutral).** These questions ask for specific figures, narrow time windows, or segment rankings. Naive refused 5/5 (no parametric memory for these specifics). RAG split: when retrieval found the right filing (Best Buy cash, MGM EBITDAR), the answer was correct and well-cited; when retrieval found the wrong filing of the right issuer (Pfizer 2021 instead of 2023, JPM 2022 Q2 instead of 2021 Q1), the answer was confidently wrong with a citation that looks legitimate. **The 50/50 split on novel-generated is the key finding**: RAG either delivers cleanly or fails dangerously, with little middle ground.

**Hypothesis for the asymmetry.**

Domain-relevant questions have a built-in safety net because they're qualitative; even when retrieval is poor, the model can fall back on general domain knowledge ("banks don't use gross margin", "telecoms are capital-intensive") that's correct often enough to cover the gap.

Novel-generated questions have no such safety net: the answer is a specific fact in a specific filing, and either retrieval surfaces the right filing or it does not.

When retrieval misses, the generator has nothing safe to fall back to, but the prompt doesn't stop it from anchoring to whatever chunks it does have, which is how we get confident-with-wrong-citation.


## Task 6 - Evaluation

Three independent measures, all automated end-to-end:

1. **Correctness** (full filtered dataset, 100 rows): LLM-as-judge with `deepseek-ai/DeepSeek-V3.2` (different model family from the generator to reduce self-preference bias). Binary verdict; only the aggregate is reported.
2. **Faithfulness** (first 20 rows by `financebench_id`): Ragas `Faithfulness`. The 20-row cap is a token-cost decision (Ragas issues several LLM calls per sample), not a manual-review one. Sync `OpenAI`-backed client wrapped via `LangchainLLMWrapper(ChatOpenAI(...))`; calls `.score(...)` with a `.single_turn_score(...)` fallback.
3. **Page-hit@k** (full filtered dataset): retrieve once at `k=5`, count a hit at each of `k ∈ {1, 3, 5}` if any retrieved chunk lands on a labeled evidence page for the right document.

The Task 6 numbers are the **baseline row** that every Task 7 experiment is compared against.

In [17]:
# Run answer_with_rag on the full 100-row filtered dataset, k=4. Cache the answers and
# retrieved chunks to disk so correctness, faithfulness, and page-hit@k can all reuse them
# without re-calling the generator. Re-run this cell after deleting the cache file to refresh.

RAG_CACHE_PATH = ARTIFACTS_DIR / "rag_outputs_full.json"

if RAG_CACHE_PATH.exists():
    with open(RAG_CACHE_PATH) as f:
        rag_outputs_full = json.load(f)
    print(f"Loaded cached RAG outputs from {RAG_CACHE_PATH.name} ({len(rag_outputs_full)} rows).")
else:
    rag_outputs_full = []
    for fbid, q in tqdm(
        list(zip(df["financebench_id"], df["question"])),
        desc="RAG full dataset",
    ):
        result = answer_with_rag(q, k=4)
        rag_outputs_full.append({
            "financebench_id": fbid,
            "question": q,
            "answer": result["answer"],
            "retrieved_chunks": result["retrieved_chunks"],
        })
    with open(RAG_CACHE_PATH, "w") as f:
        json.dump(rag_outputs_full, f, indent=2)
    print(f"Wrote {len(rag_outputs_full)} RAG outputs to {RAG_CACHE_PATH.name}.")

RAG full dataset: 100%|██████████| 100/100 [04:35<00:00,  2.76s/it]

Wrote 100 RAG outputs to rag_outputs_full.json.


In [18]:
# Page-hit@k computed by retrieving once at k=5 and slicing the result for k=1, 3, and 5.
# This is exact (not an approximation): FAISS similarity_search returns chunks strictly
# ordered by L2 distance, so similarity_search(query, k=5)[:3] is identical to
# similarity_search(query, k=3) and [:1] is identical to k=1. One retrieval call per
# question instead of three, with no correctness loss.
#
# Match is on (doc_name, page_number) against the row's labeled evidence pages: a chunk
# from a different document that happens to share a page number with the evidence is not
# a real hit. Single-doc-per-question (verified in Task 3) means restricting to the row's
# doc_name does not exclude any legitimate matches.

evidence_by_id = {}
for _, row in df.iterrows():
    pairs = set()
    for ev in row["evidence"]:
        pairs.add((ev["doc_name"], ev["evidence_page_num"]))
    evidence_by_id[row["financebench_id"]] = pairs

page_hit_records = []
for fbid, q in tqdm(
    list(zip(df["financebench_id"], df["question"])),
    desc="page-hit@k (k=5 once, sliced)",
):
    chunks_top5 = vectorstore.similarity_search(q, k=5)
    pairs_in_order = [
        (c.metadata["doc_name"], c.metadata["page_number"]) for c in chunks_top5
    ]
    evidence_pairs = evidence_by_id[fbid]
    page_hit_records.append({
        "financebench_id": fbid,
        "page_hit_at_1": int(any(p in evidence_pairs for p in pairs_in_order[:1])),
        "page_hit_at_3": int(any(p in evidence_pairs for p in pairs_in_order[:3])),
        "page_hit_at_5": int(any(p in evidence_pairs for p in pairs_in_order[:5])),
    })

page_hit_df = pd.DataFrame(page_hit_records)
print(
    "page_hit means:",
    page_hit_df[["page_hit_at_1", "page_hit_at_3", "page_hit_at_5"]].mean().round(3).to_dict(),
)

page-hit@k (k=5 once, sliced): 100%|██████████| 100/100 [00:03<00:00, 33.05it/s]


page_hit means: {'page_hit_at_1': 0.2, 'page_hit_at_3': 0.33, 'page_hit_at_5': 0.4}


In [23]:
# Correctness judge: LLM-as-judge over the full dataset using DeepSeek-V3.2 (different
# model family from the Llama-3.3 generator to reduce self-preference bias). Binary verdict
# plus a one-line reason kept on the row for spot-checks; only the aggregate is reported.

JUDGE_SYSTEM_PROMPT = (
    "You are a grader for a financial QA system. You will be given a question, a "
    "model answer, and the ground-truth answer. Decide whether the model answer is "
    "factually correct relative to the ground truth.\n"
    "- Numbers must match the ground truth within +/-2% relative error (or to the same "
    "precision as the ground truth, whichever is looser); units, scale (millions vs "
    "billions), and signs must match exactly.\n"
    "- Ignore document citation tags like \"(CORNING_2022_10K)\" in the model answer; "
    "grade only the substantive content.\n"
    "- For multi-part questions, every part must be answered correctly. Partial answers "
    "are INCORRECT.\n"
    "- An answer that says it cannot answer or refuses is INCORRECT unless the ground "
    "truth itself is a refusal.\n"
    "- Hedged language (\"approximately\", \"around\") is fine when the underlying "
    "number is correct.\n"
    "- Extra unrelated detail is fine if the core answer is right.\n"
    "Respond with a strict JSON object on a single line: "
    '{"correct": 0 or 1, "reason": "<one short sentence>"}.'
)

def judge_correctness(question: str, model_answer: str, ground_truth: str) -> dict:
    user = (
        f"Question: {question}\n\n"
        f"Model answer: {model_answer}\n\n"
        f"Ground truth: {ground_truth}"
    )
    raw = chat(
        [
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ],
        model=JUDGE_MODEL,
        temperature=0.0,
        max_tokens=120,
    )
    m = re.search(r"\{.*?\}", raw, re.DOTALL)
    if not m:
        return {"correct": 0, "reason": f"unparseable: {raw[:80]}"}
    try:
        obj = json.loads(m.group(0))
        return {"correct": int(obj.get("correct", 0)), "reason": str(obj.get("reason", ""))}
    except Exception as exc:
        return {"correct": 0, "reason": f"parse error: {exc}"}

gt_by_id = dict(zip(df["financebench_id"], df["answer"]))

correctness_records = []
for entry in tqdm(rag_outputs_full, desc="correctness judge"):
    fbid = entry["financebench_id"]
    verdict = judge_correctness(entry["question"], entry["answer"], gt_by_id[fbid])
    correctness_records.append({
        "financebench_id": fbid,
        "correctness": verdict["correct"],
        "correctness_reason": verdict["reason"],
    })

correctness_df = pd.DataFrame(correctness_records)
print(f"Mean correctness over {len(correctness_df)} rows: {correctness_df['correctness'].mean():.3f}")

correctness judge: 100%|██████████| 100/100 [01:56<00:00,  1.16s/it]

Mean correctness over 100 rows: 0.280


In [27]:
# Faithfulness on the first 20 rows by financebench_id. Ragas is token-heavy (multiple LLM
# calls per sample); 20 is the fixed cap across all experiments so faithfulness numbers stay
# comparable. Sync OpenAI client wrapped via ragas.llms.llm_factory (the v1.0-aligned pattern;
# replaces the deprecated LangchainLLMWrapper). Faithfulness stays on the legacy ragas.metrics
# import: the v1.0 collections class drops .single_turn_score, so until the assignment moves
# to the v1.0 call signature we keep the import that works with the .score()/.single_turn_score()
# fallback. Try .score() first; fall back to .single_turn_score() if it returns a coroutine.

from ragas.dataset_schema import SingleTurnSample
from ragas.llms import llm_factory
from ragas.metrics import Faithfulness

nebius_openai = OpenAI(api_key=NEBIUS_API_KEY, base_url=NEBIUS_BASE_URL)
ragas_llm = llm_factory(model=JUDGE_MODEL, client=nebius_openai)
faithfulness_metric = Faithfulness(llm=ragas_llm)

def score_faithfulness(sample: SingleTurnSample) -> float:
    score_fn = getattr(faithfulness_metric, "score", None)
    if score_fn is not None:
        try:
            result = score_fn(sample)
            if hasattr(result, "__await__"):
                raise TypeError("score returned coroutine - falling back")
            return float(result)
        except (AttributeError, NotImplementedError, TypeError):
            pass
    return float(faithfulness_metric.single_turn_score(sample))

FAITHFULNESS_N = 20
faithfulness_ids = (
    df.sort_values("financebench_id")["financebench_id"].tolist()[:FAITHFULNESS_N]
)
rag_by_id = {entry["financebench_id"]: entry for entry in rag_outputs_full}

faithfulness_records = []
for fbid in tqdm(faithfulness_ids, desc=f"faithfulness (first {FAITHFULNESS_N})"):
    entry = rag_by_id[fbid]
    sample = SingleTurnSample(
        user_input=entry["question"],
        response=entry["answer"],
        retrieved_contexts=[c["text"] for c in entry["retrieved_chunks"]],
    )
    score = score_faithfulness(sample)
    faithfulness_records.append({"financebench_id": fbid, "faithfulness": score})

faithfulness_df = pd.DataFrame(faithfulness_records)
print(
    f"Mean faithfulness over {len(faithfulness_df)} rows: "
    f"{faithfulness_df['faithfulness'].mean():.3f}"
)

/var/folders/6x/4qlnw24s11gdkhghyfkzpyl00000gn/T/ipykernel_12272/2937270838.py:11: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness
faithfulness (first 20): 100%|██████████| 20/20 [02:40<00:00,  8.03s/it]

Mean faithfulness over 20 rows: 0.793


In [28]:
# Assemble the evaluation xlsx and report Task 6 baseline aggregates. The xlsx filename
# includes a timestamp so reruns produce backups; rename the run you trust to
# assignment2_evaluation.xlsx (the deliverable name) once finalized. The baseline table is
# the reference row that every Task 7 experiment is compared against.

eval_df = (
    df[["financebench_id", "question"]]
    .merge(correctness_df[["financebench_id", "correctness"]], on="financebench_id")
    .merge(faithfulness_df, on="financebench_id", how="left")
    .merge(page_hit_df, on="financebench_id")
)
eval_df = eval_df[
    [
        "financebench_id",
        "question",
        "correctness",
        "faithfulness",
        "page_hit_at_1",
        "page_hit_at_3",
        "page_hit_at_5",
    ]
]

_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
eval_path = ARTIFACTS_DIR / f"assignment2_evaluation_{_ts}.xlsx"
eval_df.to_excel(eval_path, index=False)
print(f"Wrote {eval_path.name} ({len(eval_df)} rows).")

n_total = len(eval_df)
n_faith = int(eval_df["faithfulness"].notna().sum())

baseline = pd.DataFrame(
    [
        {"metric": "Correctness",   "scope": "full dataset",        "n": n_total, "score": eval_df["correctness"].mean()},
        {"metric": "Faithfulness",  "scope": "first 20 by id",      "n": n_faith, "score": eval_df["faithfulness"].mean()},
        {"metric": "Page-hit@1",    "scope": "full dataset",        "n": n_total, "score": eval_df["page_hit_at_1"].mean()},
        {"metric": "Page-hit@3",    "scope": "full dataset",        "n": n_total, "score": eval_df["page_hit_at_3"].mean()},
        {"metric": "Page-hit@5",    "scope": "full dataset",        "n": n_total, "score": eval_df["page_hit_at_5"].mean()},
    ]
)
baseline["score"] = baseline["score"].round(3)
print(f"\nTask 6 baseline (k=4, chunk_size=1000, no reranker):")
baseline.style.hide(axis="index").format({"score": "{:.3f}"})

Wrote assignment2_evaluation_20260428_034837.xlsx (100 rows).

Task 6 baseline (k=4, chunk_size=1000, no reranker):


metric,scope,n,score
Correctness,full dataset,100,0.280
Faithfulness,first 20 by id,20,0.793
Page-hit@1,full dataset,100,0.200
Page-hit@3,full dataset,100,0.330
Page-hit@5,full dataset,100,0.400


## Task 7 - Improvement Cycles

Three controlled experiments, one variable each. Baseline (k=4, chunk_size=1000, no reranker) is the reference row. Each experiment re-runs all three measures (correctness on full dataset, faithfulness on first 20 by id, page-hit@1/3/5 on full dataset) so movement on each axis is directly attributable to the variable that changed.

- **E1.** k sweep over {1, 3, 5, 8}. Hypothesis: more context helps the generator extract the right answer when retrieval is noisy, up to the point where the relevant chunk gets diluted.
- **E2.** Chunk size over {300, 2000} (1000 is baseline). Hypothesis: smaller chunks improve precision per chunk; larger chunks improve recall of locally-related facts.
- **E3.** Add `BAAI/bge-reranker-base` (FAISS top-20 -> cross-encoder rerank to top-4). Hypothesis: page-hit@k rises because the cross-encoder reorders the shortlist toward the truly relevant chunks; correctness follows.


In [32]:
# Task 7 helper. run_experiment(name, change, experiment_fn, ...) drives one variant
# end-to-end and returns a single row for the cycles xlsx. The experiment_fn isolates the
# only thing that varies between experiments: how the chunks are produced for a given query.
# Returned per-row page-hit records are kept in PER_ROW_PAGEHIT for the bonus task.
#
# Page-hit is reported at k in {1, 3, 5, 8} - one column per tested k value across E1 -
# computed from a single top-8 retrieval per question (FAISS is deterministic so slicing
# is exact for the smaller depths).

PER_ROW_PAGEHIT: dict[str, list[dict]] = {}

def _generate_given_chunks(query: str, chunks: list) -> str:
    """RAG generator with retrieval factored out (chunks supplied by the caller)."""
    context_block = format_context_block(chunks)
    user_message = f"Question: {query}\n\nContext:\n{context_block}"
    return chat(
        [
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        model=GENERATION_MODEL,
        temperature=0.0,
        max_tokens=500,
    )


def run_experiment(
    name: str,
    change: str,
    experiment_fn,
    n_corr: int | None = None,
    n_faith: int = 20,
) -> dict:
    """Run a single Task 7 experiment.

    experiment_fn(query) must return a dict with keys:
      - "answer": str (model output)
      - "chunks_for_gen": list of LangChain Documents the generator saw
      - "chunks_top8": list of 8 LangChain Documents in rank order, used for page-hit@k
    n_corr: rows used for correctness (full dataset if None). Smoke tests pass small int.
    n_faith: rows used for faithfulness (always first n_faith by financebench_id).
    """
    n_corr = len(df) if n_corr is None else n_corr
    eval_subset = df.sort_values("financebench_id").head(n_corr).reset_index(drop=True)

    # 1. Per-question RAG (retrieve + generate via experiment_fn)
    rag_records: list[dict] = []
    for fbid, q in tqdm(
        list(zip(eval_subset["financebench_id"], eval_subset["question"])),
        desc=f"{name} RAG",
    ):
        out = experiment_fn(q)
        rag_records.append({
            "financebench_id": fbid,
            "question": q,
            "answer": out["answer"],
            "chunks_for_gen": out["chunks_for_gen"],
            "chunks_top8": out["chunks_top8"],
        })

    # 2. Page-hit@k from chunks_top8 (k in {1, 3, 5, 8})
    page_hit_rows = []
    for r in rag_records:
        evidence_pairs = evidence_by_id.get(r["financebench_id"], set())
        pairs = [
            (c.metadata["doc_name"], c.metadata["page_number"]) for c in r["chunks_top8"]
        ]
        page_hit_rows.append({
            "financebench_id": r["financebench_id"],
            "page_hit_at_1": int(any(p in evidence_pairs for p in pairs[:1])),
            "page_hit_at_3": int(any(p in evidence_pairs for p in pairs[:3])),
            "page_hit_at_5": int(any(p in evidence_pairs for p in pairs[:5])),
            "page_hit_at_8": int(any(p in evidence_pairs for p in pairs[:8])),
        })
    PER_ROW_PAGEHIT[name] = page_hit_rows
    page_hit_df_x = pd.DataFrame(page_hit_rows)

    # 3. Correctness judge
    gt_local = dict(zip(df["financebench_id"], df["answer"]))
    corr_scores = []
    for r in tqdm(rag_records, desc=f"{name} judge"):
        v = judge_correctness(r["question"], r["answer"], gt_local[r["financebench_id"]])
        corr_scores.append(v["correct"])

    # 4. Faithfulness on first n_faith rows by financebench_id
    faith_rows = sorted(rag_records, key=lambda x: x["financebench_id"])[:n_faith]
    faith_scores = []
    for r in tqdm(faith_rows, desc=f"{name} faith"):
        sample = SingleTurnSample(
            user_input=r["question"],
            response=r["answer"],
            retrieved_contexts=[c.page_content for c in r["chunks_for_gen"]],
        )
        try:
            faith_scores.append(score_faithfulness(sample))
        except Exception as exc:
            print(f"  faith failed for {r['financebench_id']}: {exc}")
            faith_scores.append(float("nan"))

    return {
        "experiment": name,
        "change": change,
        "correctness": float(pd.Series(corr_scores).mean()),
        "faithfulness": float(pd.Series(faith_scores).mean()),
        "page_hit_at_1": float(page_hit_df_x["page_hit_at_1"].mean()),
        "page_hit_at_3": float(page_hit_df_x["page_hit_at_3"].mean()),
        "page_hit_at_5": float(page_hit_df_x["page_hit_at_5"].mean()),
        "page_hit_at_8": float(page_hit_df_x["page_hit_at_8"].mean()),
    }


def build_or_load_index_at_size(chunk_size: int, chunk_overlap: int = 150):
    """Build (or load cached) FAISS index at the given chunk_size. Same loader/embedder/splitter
    pattern as the baseline cell; persisted under indices/faiss_chunk{size} so each variant
    survives kernel restarts."""
    idx_dir = REPO_ROOT / "indices" / f"faiss_chunk{chunk_size}"
    if idx_dir.exists():
        print(f"Loading cached index from {idx_dir.relative_to(REPO_ROOT)}")
        return FAISS.load_local(str(idx_dir), embeddings, allow_dangerous_deserialization=True)
    print(f"Building FAISS index at {idx_dir.relative_to(REPO_ROOT)} (chunk_size={chunk_size})")
    doc_meta = (
        df.drop_duplicates("doc_name")
          .set_index("doc_name")[["company", "doc_period"]]
          .to_dict("index")
    )
    page_docs = []
    for name in tqdm(doc_names, desc=f"Loading PDFs (cs={chunk_size})"):
        loader = PyPDFLoader(str(PDF_DIR / f"{name}.pdf"))
        for i, page in enumerate(loader.load()):
            page.metadata = {
                "doc_name": name,
                "company": doc_meta[name]["company"],
                "doc_period": doc_meta[name]["doc_period"],
                "page_number": i,
            }
            page_docs.append(page)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    chunks_local = splitter.split_documents(page_docs)
    print(f"Split into {len(chunks_local)} chunks (size={chunk_size}, overlap={chunk_overlap}).")
    vs = FAISS.from_documents(chunks_local, embeddings)
    idx_dir.parent.mkdir(parents=True, exist_ok=True)
    vs.save_local(str(idx_dir))
    print(f"Saved index to {idx_dir.relative_to(REPO_ROOT)}")
    return vs


print("Task 7 helpers defined: run_experiment, _generate_given_chunks, build_or_load_index_at_size")


Task 7 helpers defined: run_experiment, _generate_given_chunks, build_or_load_index_at_size


### Smoke test

Tiny end-to-end run before the overnight loop. Validates that `run_experiment` wires up correctly: 3 rows for correctness, 3 rows for faithfulness, k=1 for fastest retrieval. Expect numeric outputs in the right ranges; the actual values are not interesting at n=3.


In [33]:
# Smoke test: 3 rows correctness, 3 rows faithfulness, k=1. ~2-3 minutes total.
# If this returns sensible numbers, the helper is wired correctly and the full loop is safe to run.

def _smoke_fn(q):
    top8 = vectorstore.similarity_search(q, k=8)
    gen_chunks = top8[:1]
    return {
        "answer": _generate_given_chunks(q, gen_chunks),
        "chunks_for_gen": gen_chunks,
        "chunks_top8": top8,
    }

smoke_row = run_experiment(
    name="smoke",
    change="k=1, n_corr=3, n_faith=3",
    experiment_fn=_smoke_fn,
    n_corr=3,
    n_faith=3,
)
print("\nsmoke row:")
for k, v in smoke_row.items():
    print(f"  {k}: {v}")


smoke faith: 100%|██████████| 3/3 [00:24<00:00,  8.08s/it]


smoke row:
  experiment: smoke
  change: k=1, n_corr=3, n_faith=3
  correctness: 0.0
  faithfulness: 0.85
  page_hit_at_1: 0.3333333333333333
  page_hit_at_3: 0.3333333333333333
  page_hit_at_5: 0.3333333333333333
  page_hit_at_8: 0.6666666666666666


### E1 - k sweep over {1, 3, 5, 8}

**Hypothesis.** Increasing k gives the generator more chances to see the relevant chunk, raising correctness; past a point, the prompt fills with unrelated text and the generator degrades. Page-hit@k for the generator's k is the upper bound on correctness; faithfulness should stay roughly flat because all chunks come from the same retriever.

**Note on page-hit columns.** The cycles xlsx reports four page-hit columns - one per tested k from E1: `page_hit_at_1`, `page_hit_at_3`, `page_hit_at_5`, `page_hit_at_8`. All four are computed from a single top-8 retrieval per question (FAISS is deterministic, so slicing top-8 yields exact lower-k results). Across the E1 rows the retriever and index are unchanged, so all four page-hit columns are constants - what moves is correctness. The relevant ceiling for each E1 row is the page-hit column at the matching k: `page_hit_at_8` for `E1_k8`, `page_hit_at_5` for `E1_k5`, etc.


In [31]:
# E1: k sweep. Run k in {1, 3, 5, 8}; baseline is k=4 (already in the cycles xlsx as row 0).
# One FAISS retrieval per question at k=max(8, k_gen); top-8 used for page-hit, top-k_gen used
# for the generator. For k_gen <= 8 this is one call; the helper handles all four hit depths.

def make_e1_fn(k_gen: int):
    def _fn(q):
        k_max = max(k_gen, 8)
        chunks = vectorstore.similarity_search(q, k=k_max)
        gen_chunks = chunks[:k_gen]
        top8 = chunks[:8]
        return {
            "answer": _generate_given_chunks(q, gen_chunks),
            "chunks_for_gen": gen_chunks,
            "chunks_top8": top8,
        }
    return _fn

e1_rows = []
for k_gen in [1, 3, 5, 8]:
    row = run_experiment(
        name=f"E1_k{k_gen}",
        change=f"retrieval depth k={k_gen} (baseline k=4)",
        experiment_fn=make_e1_fn(k_gen),
    )
    print(f"E1 k={k_gen}: {row}")
    e1_rows.append(row)


E1_k1 faith: 100%|██████████| 20/20 [02:28<00:00,  7.42s/it]


E1 k=1: {'experiment': 'E1_k1', 'change': 'retrieval depth k=1 (baseline k=4)', 'correctness': 0.15, 'faithfulness': 0.7288888888888889, 'page_hit_at_1': 0.2, 'page_hit_at_3': 0.33, 'page_hit_at_5': 0.4}


E1_k3 faith: 100%|██████████| 20/20 [02:38<00:00,  7.94s/it]


E1 k=3: {'experiment': 'E1_k3', 'change': 'retrieval depth k=3 (baseline k=4)', 'correctness': 0.24, 'faithfulness': 0.8094444444444445, 'page_hit_at_1': 0.2, 'page_hit_at_3': 0.33, 'page_hit_at_5': 0.4}


E1_k5 faith: 100%|██████████| 20/20 [02:45<00:00,  8.30s/it]


E1 k=5: {'experiment': 'E1_k5', 'change': 'retrieval depth k=5 (baseline k=4)', 'correctness': 0.3, 'faithfulness': 0.7791666666666666, 'page_hit_at_1': 0.2, 'page_hit_at_3': 0.33, 'page_hit_at_5': 0.4}


E1_k8 faith: 100%|██████████| 20/20 [03:03<00:00,  9.18s/it]

E1 k=8: {'experiment': 'E1_k8', 'change': 'retrieval depth k=8 (baseline k=4)', 'correctness': 0.35, 'faithfulness': 0.7286904761904762, 'page_hit_at_1': 0.2, 'page_hit_at_3': 0.33, 'page_hit_at_5': 0.4}


### E2 - chunk size over {300, 2000}

**Hypothesis.** chunk_size=300 produces tighter, higher-precision chunks (one fact per chunk) but loses cross-paragraph context. chunk_size=2000 keeps full sections together (better recall of nearby facts) but each chunk is more diluted from the embedder's perspective. The baseline 1000 is the standard middle ground.

Each chunk size requires a fresh FAISS build under `indices/faiss_chunk{size}`. Built once and cached; subsequent runs reuse the saved index.


In [ ]:
# E2: chunk_size in {300, 2000}. Each requires re-embedding the corpus once
# (~5-10 min per size on CPU); subsequent runs hit the on-disk cache.

E2_CHUNK_SIZES = [300, 2000]
e2_indices: dict[int, object] = {}
for cs in E2_CHUNK_SIZES:
    e2_indices[cs] = build_or_load_index_at_size(cs)
    print(f"  chunk_size={cs}: {e2_indices[cs].index.ntotal} vectors")

def make_e2_fn(vs):
    def _fn(q):
        top8 = vs.similarity_search(q, k=8)
        gen_chunks = top8[:4]  # match baseline k=4
        return {
            "answer": _generate_given_chunks(q, gen_chunks),
            "chunks_for_gen": gen_chunks,
            "chunks_top8": top8,
        }
    return _fn

e2_rows = []
for cs in E2_CHUNK_SIZES:
    row = run_experiment(
        name=f"E2_chunk{cs}",
        change=f"chunk_size={cs} (baseline 1000)",
        experiment_fn=make_e2_fn(e2_indices[cs]),
    )
    print(f"E2 chunk_size={cs}: {row}")
    e2_rows.append(row)


### E3 - cross-encoder reranker

**Hypothesis.** FAISS's bi-encoder retrieval is fast but imprecise (the query and chunk are encoded separately). A cross-encoder scores each `(query, chunk)` pair jointly, so reordering the top-20 should push truly relevant chunks higher. Page-hit@1 should rise the most (top-1 reordered first); correctness should follow because the generator sees better evidence; faithfulness should stay flat or rise slightly.

`BAAI/bge-reranker-base` is small (~280 MB) and downloads to the HuggingFace cache on first use.


In [ ]:
# E3: retrieve FAISS top-20, rerank with BAAI/bge-reranker-base, keep top-8 for page-hit
# and top-4 for the generator (matches baseline k=4).

from sentence_transformers import CrossEncoder

RERANKER_NAME = "BAAI/bge-reranker-base"
print(f"Loading {RERANKER_NAME} (downloads ~280 MB on first use)...")
reranker = CrossEncoder(RERANKER_NAME)
print("Reranker loaded.")

def _e3_fn(q):
    top20 = vectorstore.similarity_search(q, k=20)
    pairs = [[q, c.page_content] for c in top20]
    scores = reranker.predict(pairs)
    # higher score = more relevant
    reranked = [c for _, c in sorted(zip(scores, top20), key=lambda x: -x[0])]
    top8 = reranked[:8]
    gen_chunks = reranked[:4]
    return {
        "answer": _generate_given_chunks(q, gen_chunks),
        "chunks_for_gen": gen_chunks,
        "chunks_top8": top8,
    }

e3_row = run_experiment(
    name="E3_reranker",
    change="FAISS top-20 -> bge-reranker-base -> top-4",
    experiment_fn=_e3_fn,
)
print(f"\nE3 reranker: {e3_row}")


In [ ]:
# Assemble assignment2_improvement_cycles.xlsx. Row 0 = Task 6 baseline; subsequent
# rows = E1/E2/E3 experiments. One page-hit column per tested k from E1 (1, 3, 5, 8).
# Filename includes a timestamp; rename the run you trust to assignment2_improvement_cycles.xlsx.
#
# Baseline page_hit_at_8 was not computed in Task 6 (which only reports {1, 3, 5}); we
# backfill it here with a quick deterministic FAISS pass (no LLM calls, ~30 seconds).

baseline_pagehit_at_8_records = []
for fbid, q in tqdm(
    list(zip(df["financebench_id"], df["question"])),
    desc="baseline page_hit@8 backfill",
):
    top8 = vectorstore.similarity_search(q, k=8)
    pairs = [(c.metadata["doc_name"], c.metadata["page_number"]) for c in top8]
    evidence_pairs = evidence_by_id[fbid]
    baseline_pagehit_at_8_records.append({
        "financebench_id": fbid,
        "page_hit_at_8": int(any(p in evidence_pairs for p in pairs)),
    })
baseline_pagehit_at_8 = float(
    pd.DataFrame(baseline_pagehit_at_8_records)["page_hit_at_8"].mean()
)

baseline_row = {
    "experiment": "baseline",
    "change": "k=4, chunk_size=1000, no reranker",
    "correctness": float(eval_df["correctness"].mean()),
    "faithfulness": float(eval_df["faithfulness"].mean()),
    "page_hit_at_1": float(eval_df["page_hit_at_1"].mean()),
    "page_hit_at_3": float(eval_df["page_hit_at_3"].mean()),
    "page_hit_at_5": float(eval_df["page_hit_at_5"].mean()),
    "page_hit_at_8": baseline_pagehit_at_8,
}

cycles_rows = [baseline_row] + e1_rows + e2_rows + [e3_row]
cycles_df = pd.DataFrame(cycles_rows)[
    [
        "experiment",
        "change",
        "correctness",
        "faithfulness",
        "page_hit_at_1",
        "page_hit_at_3",
        "page_hit_at_5",
        "page_hit_at_8",
    ]
]

_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
cycles_path = ARTIFACTS_DIR / f"assignment2_improvement_cycles_{_ts}.xlsx"
cycles_df.to_excel(cycles_path, index=False)
print(f"Wrote {cycles_path.name} ({len(cycles_df)} rows).")

print("\nTask 7 summary:")
score_cols = [
    "correctness", "faithfulness",
    "page_hit_at_1", "page_hit_at_3", "page_hit_at_5", "page_hit_at_8",
]
cycles_df_display = cycles_df.copy()
cycles_df_display[score_cols] = cycles_df_display[score_cols].round(3)
cycles_df_display.style.hide(axis="index").format({c: "{:.3f}" for c in score_cols})


### Task 7 - Wrap-up

**Where the pipeline fails most.**

_(Filled in after the overnight run, based on the cycles xlsx numbers.)_ The big gap between baseline faithfulness (~0.75) and baseline correctness (~0.28) already pointed to retrieval as the bottleneck before Task 7 even ran: the generator was answering faithfully from whatever it was given, but it was being given the wrong pages. Whichever experiment moves page-hit@1 the most is the one that addresses this directly. If E3 (reranker) and E2 (chunk_size) both lift correctness while E1 (more k) does not, that confirms retrieval - not generation - is the binding constraint.

**One more week.**

_(Filled in after the overnight run.)_ Likely candidates: (a) hybrid retrieval (BM25 + dense) so exact-match terms like ticker symbols and dollar amounts are not lost in the embedder's averaging; (b) per-document filtering at retrieval time using the question's known target company, since FinanceBench questions are single-doc; (c) a reranker fine-tuned on financial QA rather than the general-purpose `bge-reranker-base`.


## Bonus - Multi-Scale Chunking

Reuses E2's per-question page-hit data (no additional compute). Compares page-hit@5 across chunk sizes {300, 1000, 2000} and reports the disagreement rate: how often the per-question best chunk size differs from the overall winner. A low disagreement rate means one chunk size dominates; a high rate means the best size is query-dependent and a static choice leaves performance on the table.


In [ ]:
# Bonus: page-hit@5 per chunk size + disagreement rate.
# Pulls per-row page-hit data from PER_ROW_PAGEHIT (E2) plus the baseline page_hit_records (chunk_size=1000).

bonus_per_row = {
    300: PER_ROW_PAGEHIT["E2_chunk300"],
    1000: page_hit_records,  # baseline, from cell 28
    2000: PER_ROW_PAGEHIT["E2_chunk2000"],
}

bonus_summary = pd.DataFrame([
    {"chunk_size": cs, "page_hit_at_5": pd.DataFrame(rows)["page_hit_at_5"].mean()}
    for cs, rows in bonus_per_row.items()
])
bonus_summary["page_hit_at_5"] = bonus_summary["page_hit_at_5"].round(3)
print("page_hit@5 per chunk size:")
print(bonus_summary.to_string(index=False))

# Per-question hit vector across chunk sizes
ids = [r["financebench_id"] for r in bonus_per_row[1000]]
hits_by_size = {
    cs: {r["financebench_id"]: r["page_hit_at_5"] for r in rows}
    for cs, rows in bonus_per_row.items()
}

# Overall winner = chunk_size with highest mean page_hit@5
overall_winner = int(bonus_summary.sort_values("page_hit_at_5", ascending=False).iloc[0]["chunk_size"])

# Per-question best = any chunk_size that hit on this question (ties broken by overall winner)
disagreements = 0
considered = 0
for fbid in ids:
    sizes_that_hit = [cs for cs in bonus_per_row.keys() if hits_by_size[cs][fbid] == 1]
    if not sizes_that_hit:
        continue  # no chunk size hit; not a disagreement signal
    considered += 1
    if overall_winner not in sizes_that_hit:
        disagreements += 1

disagreement_rate = disagreements / considered if considered else float("nan")
print(f"\nOverall winner: chunk_size={overall_winner} (page_hit@5 = {bonus_summary[bonus_summary['chunk_size']==overall_winner]['page_hit_at_5'].iloc[0]:.3f})")
print(f"Questions with at least one hit across the three sizes: {considered}/{len(ids)}")
print(f"Disagreement rate (best size != overall winner): {disagreements}/{considered} = {disagreement_rate:.3f}")


### Bonus - Discussion

_(Filled in after the overnight run, based on the disagreement rate above.)_

- **If disagreement rate is low (< 0.2):** one chunk size dominates and the choice transfers across the corpus. A static chunk_size is a defensible engineering decision.
- **If disagreement rate is moderate (0.2-0.5):** the overall winner is correct on average but leaves measurable accuracy on the table. A simple ensemble (retrieve from all three indices, union the top-k) would likely raise page_hit@5 several points without changing the generator.
- **If disagreement rate is high (> 0.5):** chunk size is genuinely query-dependent. Smaller chunks win for keyword-heavy lookups; larger chunks win for questions that need cross-paragraph context. A static choice is suboptimal and a multi-scale retriever is the right next step.
